# Q5 — Language Modeling (WikiText-2)

**Canonical path**: Matched 3000 train / 400 val / 400 test comparison across Trigram, LSTM, DistilGPT2.

**Runtime**: T4 GPU required for LSTM and GPT-2 cells.

> This notebook reproduces the report-facing Q5 comparison artifacts.
> For full WikiText-2 runs, see the **Exploratory** section at the bottom.

## 1. Environment Setup

In [ ]:
import os, sys, json, glob, shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
    print('Not running on Colab — skipping Drive mount.')

DRIVE_OUTPUT = '/content/drive/MyDrive/467-takehome-outputs/q5' if ON_COLAB else None
if DRIVE_OUTPUT:
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)

In [ ]:
REPO_DIR = '/content/467-takehome' if ON_COLAB else os.path.abspath('..')
if ON_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/zubeyralmaho/467-takehome.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull --ff-only
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')

In [ ]:
def save_to_drive(run_dir, tag=''):
    if not DRIVE_OUTPUT:
        return
    name = os.path.basename(run_dir) + (f'_{tag}' if tag else '')
    dest = os.path.join(DRIVE_OUTPUT, name)
    shutil.copytree(run_dir, dest, dirs_exist_ok=True)
    print(f'Saved to Drive: {dest}')

def gpu_cleanup():
    import gc
    torch.cuda.empty_cache()
    gc.collect()
    if torch.cuda.is_available():
        print(f'GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 2. Canonical Run — Matched 3000/400/400 Comparison

Reproduces the report-facing comparison: 3000 train lines / 400 val / 400 test, all 3 models on the same budget.

### 2a. Trigram (CPU)

In [ ]:
!python -m src.q5_language_modeling.main \
    --config configs/q5.yaml \
    --final-eval \
    --override \
        model.type=ngram \
        model.n=3 \
        dataset.limit_train_samples=3000 \
        dataset.limit_validation_samples=400 \
        dataset.limit_test_samples=400

In [ ]:
ngram_run = sorted(glob.glob('outputs/q5/run_*'))[-1]
save_to_drive(ngram_run, 'trigram')
print(f'Trigram run: {ngram_run}')

### 2b. LSTM (GPU)

In [ ]:
!python -m src.q5_language_modeling.main \
    --config configs/q5.yaml \
    --final-eval \
    --override \
        model.type=lstm \
        model.hidden_dim=128 \
        model.num_layers=2 \
        model.dropout=0.2 \
        model.max_epochs=8 \
        model.early_stopping_patience=2 \
        model.batch_size=32 \
        dataset.limit_train_samples=3000 \
        dataset.limit_validation_samples=400 \
        dataset.limit_test_samples=400

In [ ]:
lstm_run = sorted(glob.glob('outputs/q5/run_*'))[-1]
save_to_drive(lstm_run, 'lstm')
gpu_cleanup()

### 2c. DistilGPT2 (GPU)

In [ ]:
!python -m src.q5_language_modeling.main \
    --config configs/q5.yaml \
    --final-eval \
    --override \
        model.type=gpt2 \
        model.model_name=distilgpt2 \
        model.eval_batch_size=8 \
        model.max_input_length=256 \
        dataset.limit_train_samples=3000 \
        dataset.limit_validation_samples=400 \
        dataset.limit_test_samples=400

In [ ]:
gpt2_run = sorted(glob.glob('outputs/q5/run_*'))[-1]
save_to_drive(gpt2_run, 'gpt2')
gpu_cleanup()

## 3. Generate Report Comparison Artifact

The Q5 report summary and figure are produced by dedicated scripts.

In [ ]:
print('Canonical Q5 runs:')
print(f'  Trigram:    {ngram_run}')
print(f'  LSTM:       {lstm_run}')
print(f'  DistilGPT2: {gpt2_run}')
print()
print('To regenerate report artifacts:')
print('  python scripts/q5_report_summary.py')
print('  python scripts/report_comparison_figures.py')

## 4. Results Summary

In [ ]:
print('=== Q5 Canonical Results ===')
for label, run_dir in [('Trigram', ngram_run), ('LSTM', lstm_run), ('DistilGPT2', gpt2_run)]:
    metrics_file = os.path.join(run_dir, 'metrics.json')
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            m = json.load(f)
        print(f'\n--- {label} ({os.path.basename(run_dir)}) ---')
        print(json.dumps(m, indent=2, default=str)[:1500])
    else:
        print(f'\n--- {label}: metrics.json not found ---')

---
## (Exploratory) Full WikiText-2 Runs

These runs use uncapped WikiText-2 and are **not** part of the canonical report comparison. Uncomment to run.

In [ ]:
# # --- N-gram full ---
# !python -m src.q5_language_modeling.main \
#     --config configs/q5.yaml --final-eval \
#     --override model.type=ngram \
#         dataset.limit_train_samples=null \
#         dataset.limit_validation_samples=null \
#         dataset.limit_test_samples=null

# # --- LSTM full, bigger model ---
# !python -m src.q5_language_modeling.main \
#     --config configs/q5.yaml --final-eval \
#     --override model.type=lstm \
#         model.hidden_dim=256 model.num_layers=2 \
#         model.dropout=0.3 model.max_epochs=15 \
#         dataset.limit_train_samples=null \
#         dataset.limit_validation_samples=null \
#         dataset.limit_test_samples=null

# # --- GPT-2 full ---
# !python -m src.q5_language_modeling.main \
#     --config configs/q5.yaml --final-eval \
#     --override model.type=gpt2 model.model_name=distilgpt2 \
#         dataset.limit_train_samples=null \
#         dataset.limit_validation_samples=null \
#         dataset.limit_test_samples=null

# Q5 — Language Modeling (WikiText-2)
**Canonical comparison:** Trigram add-k (CPU) -> LSTM (GPU) -> distilGPT2 (GPU)

**Canonical split:** 3000 train / 400 validation / 400 test lines

**Report refresh chain:** three matched runs -> `scripts/q5_report_summary.py` -> `scripts/report_comparison_figures.py`

**Runtime:** T4 GPU recommended for the LSTM and distilGPT2 cells

**Exploratory note:** Larger null-cap or full-data reruns are optional follow-up work and should be treated as exploratory rather than direct replacements for the current report comparison.

## 1. Environment Setup

In [ ]:
import os, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/467-takehome-outputs/q5'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print('Drive mounted, output dir ready.')

In [ ]:
REPO_DIR = '/content/467-takehome'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/zubeyralmaho/467-takehome.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import json
import re
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PYTHON = sys.executable
REPO_ROOT = Path.cwd()
Q5_CANONICAL_SPLIT_OVERRIDES = [
    'dataset.limit_train_samples=3000',
    'dataset.limit_validation_samples=400',
    'dataset.limit_test_samples=400',
]
Q5_REFERENCE_ARTIFACTS = {
    'ngram': 'outputs/q5/run_20260413_202258',
    'lstm': 'outputs/q5/run_20260413_211945',
    'gpt2': 'outputs/q5/run_20260413_213856',
    'summary': 'outputs/q5/run_20260413_214837',
}
Q5_RUN_DIRS = {}
Q5_SUMMARY_DIR = None


def run_logged_command(cmd):
    print('$ ' + shlex.join(cmd))
    output_lines = []
    with subprocess.Popen(
        cmd,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            output_lines.append(line)
        return_code = process.wait()
    output = ''.join(output_lines)
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd, output=output)
    return output


def extract_saved_path(command_output, prefix):
    match = re.search(rf'{re.escape(prefix)} (.+)', command_output)
    if not match:
        raise RuntimeError(f"Could not find '{prefix}' in command output.")
    return Path(match.group(1).strip()).resolve()


def run_q5_model(model_type, extra_overrides=None):
    overrides = [f'model.type={model_type}', *Q5_CANONICAL_SPLIT_OVERRIDES, *(extra_overrides or [])]
    cmd = [
        PYTHON,
        '-m',
        'src.q5_language_modeling.main',
        '--config',
        'configs/q5.yaml',
        '--final-eval',
        '--override',
        *overrides,
    ]
    output = run_logged_command(cmd)
    run_dir = extract_saved_path(output, 'Saved outputs to')
    Q5_RUN_DIRS[model_type] = run_dir
    print(f'Registered {model_type} run: {run_dir}')
    return run_dir


def copy_artifact_to_drive(path, label):
    artifact_path = Path(path)
    if artifact_path.is_dir():
        destination = Path(DRIVE_OUTPUT) / f'{artifact_path.name}_{label}'
        shutil.copytree(artifact_path, destination, dirs_exist_ok=True)
    else:
        destination = Path(DRIVE_OUTPUT) / f'{artifact_path.stem}_{label}{artifact_path.suffix}'
        shutil.copy2(artifact_path, destination)
    print(f'Copied {artifact_path} -> {destination}')
    return destination


def build_q5_summary():
    global Q5_SUMMARY_DIR
    missing_models = {'ngram', 'lstm', 'gpt2'} - set(Q5_RUN_DIRS)
    if missing_models:
        raise RuntimeError(f'Run these models first: {sorted(missing_models)}')
    cmd = [
        PYTHON,
        'scripts/q5_report_summary.py',
        '--run',
        str(Q5_RUN_DIRS['ngram']),
        '--run',
        str(Q5_RUN_DIRS['lstm']),
        '--run',
        str(Q5_RUN_DIRS['gpt2']),
    ]
    output = run_logged_command(cmd)
    Q5_SUMMARY_DIR = extract_saved_path(output, 'Saved Q5 report summary outputs to')
    print(f'Registered Q5 summary artifact: {Q5_SUMMARY_DIR}')
    return Q5_SUMMARY_DIR


def refresh_q5_report_figure(summary_dir=None):
    if summary_dir is None and Q5_SUMMARY_DIR is None:
        raise RuntimeError('Build the Q5 summary artifact before refreshing the report figure.')
    summary_root = Path(summary_dir or Q5_SUMMARY_DIR)
    cmd = [
        PYTHON,
        'scripts/report_comparison_figures.py',
        '--q5-summary',
        str(summary_root / 'q5_report_summary.json'),
    ]
    run_logged_command(cmd)
    figure_path = REPO_ROOT / 'report' / 'figures' / 'q5' / 'perplexity_comparison.png'
    print(f'Refreshed Q5 report figure: {figure_path}')
    return figure_path


print('Canonical report split: 3000 train / 400 validation / 400 test lines')
print(json.dumps(Q5_REFERENCE_ARTIFACTS, indent=2))

## 2. Run Canonical Trigram Baseline (CPU)
This default cell reproduces the trigram reference used in the matched 3000/400/400 Q5 comparison.

In [ ]:
ngram_run = run_q5_model(
    'ngram',
    extra_overrides=[
        'model.n=3',
        'model.smoothing=add_k',
        'model.alpha=0.1',
        'model.min_token_frequency=2',
    ],
)

In [ ]:
ngram_drive_copy = copy_artifact_to_drive(ngram_run, 'trigram_canonical')

## 3. Run Canonical LSTM Baseline (GPU)
This path matches the capped LSTM comparison run that feeds the current report summary chain.

In [ ]:
lstm_run = run_q5_model(
    'lstm',
    extra_overrides=[
        'model.max_vocab_size=10000',
        'model.embedding_dim=128',
        'model.hidden_dim=128',
        'model.num_layers=2',
        'model.dropout=0.2',
        'model.batch_size=32',
        'model.learning_rate=0.001',
        'model.max_epochs=3',
        'model.early_stopping_patience=2',
        'model.max_seq_length=35',
        'model.tie_weights=true',
        'model.gradient_clip=1.0',
    ],
)

In [ ]:
lstm_drive_copy = copy_artifact_to_drive(lstm_run, 'lstm_canonical')

In [ ]:
import gc

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
allocated_gb = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f'GPU memory freed. Allocated: {allocated_gb:.2f} GB')

## 4. Run Canonical distilGPT2 Baseline (GPU)
This default cell reproduces the practical GPT-style baseline used in the current matched Q5 report comparison.

In [ ]:
gpt2_run = run_q5_model(
    'gpt2',
    extra_overrides=[
        'model.model_name=distilgpt2',
        'model.max_input_length=256',
        'model.eval_batch_size=8',
        'model.top_k=50',
    ],
)

In [ ]:
gpt2_drive_copy = copy_artifact_to_drive(gpt2_run, 'gpt2_canonical')

## 5. Build the Report-Ready Summary Chain
After the three matched runs finish, build a fresh Q5 summary artifact and refresh the Q5 report figure from that summary JSON.

If you later remove the sample caps for a larger rerun, treat that output as exploratory and generate a separate summary instead of overwriting the canonical comparison chain.

In [ ]:
q5_summary_dir = build_q5_summary()
q5_summary_drive_copy = copy_artifact_to_drive(q5_summary_dir, 'report_summary')
q5_figure_path = refresh_q5_report_figure(q5_summary_dir)
q5_figure_drive_copy = copy_artifact_to_drive(q5_figure_path, 'report_figure')

print('\n=== Current notebook outputs ===')
print(
    json.dumps(
        {
            'runs': {model: str(path) for model, path in Q5_RUN_DIRS.items()},
            'summary': str(q5_summary_dir),
            'report_figure': str(q5_figure_path),
            'drive_copies': {
                'ngram': str(ngram_drive_copy),
                'lstm': str(lstm_drive_copy),
                'gpt2': str(gpt2_drive_copy),
                'summary': str(q5_summary_drive_copy),
                'report_figure': str(q5_figure_drive_copy),
            },
        },
        indent=2,
    )
)

print('\nReference artifacts used by the current report:')
print(json.dumps(Q5_REFERENCE_ARTIFACTS, indent=2))
print(
    '\nExploratory note: larger null-cap or full-data Q5 reruns should be summarized separately and not treated as direct replacements for the matched report comparison.'
)